# Unit 6 · Logistic regression

You are reviewing a spam filter. Given the percentage of email characters that are exclamation marks, what Spam probability will a model assign? When should that probability trigger a flag?

This is the same UCI Spambase data as the Classification Metrics lesson. Here we fit a one-feature model so that you can trace every score by hand. Follow the cells in order. [Open the interactive Unit 6 lesson](https://sharanharsoor.github.io/ml-learning-lab/ml-foundations/#unit-6)

## 1. Open and split the records

Each row comes from one email. Column `fields[51]` in Python (the 52nd source column) is the percentage of characters that are exclamation marks. An input of 0.428 means 0.428% of characters. The label column stores Spam as 1 and Legitimate as 0. The original messages are unavailable.

Give each partition one purpose. Fit coefficients on training rows, pick a cutoff on validation rows, and evaluate that fixed choice on final test rows. We hash the full raw feature profile into ten reproducible buckets so identical profiles always go to the same partition. The model itself only sees the exclamation mark feature.

Spambase was collected by Mark Hopkins, Erik Reeber, George Forman, and Jaap Suermondt. [UCI dataset and feature definitions](https://doi.org/10.24432/C53G6X) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
import csv
import hashlib
import io
import math
import os
from pathlib import Path
from statistics import mean
from urllib.request import urlopen

source_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"
local_file = os.environ.get("SPAMBASE_DATA")
if local_file:
    source = Path(local_file).read_text(encoding="ascii")
else:
    with urlopen(source_url, timeout=30) as response:
        source = response.read().decode("ascii")

splits = {part: [] for part in ("train", "validation", "test")}
for number, fields in enumerate(csv.reader(io.StringIO(source)), start=1):
    profile = "37:" + ",".join(fields[:-1])
    digest = hashlib.sha256(profile.encode("ascii")).hexdigest()
    bucket = int(digest[:8], 16) % 10
    part = "train" if bucket < 6 else "validation" if bucket < 8 else "test"
    splits[part].append((f"E{number}", float(fields[51]), int(fields[-1])))

for part, rows in splits.items():
    print(f"{part:10} {len(rows):,} emails, {sum(y for _, _, y in rows):,} Spam")

## 2. Turn a linear score into a probability

Unit 5 used the output of a line directly as a predicted rental count. Here the line gives a score that can be any real number. The sigmoid converts it to a probability between 0 and 1.

**Linear score** · $z = b + wx$

**Spam probability** · $p = 1/(1 + e^{-z})$

**Spam odds** · $p/(1-p) = e^z$

Raising $x$ by one percentage point adds $w$ to the score and multiplies the odds by $e^w$. The change in probability depends on where you start on the curve.

In [ ]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e_z = math.exp(z)
    return e_z / (1 + e_z)

for z in (-4, -2, 0, 2, 4):
    print(f"z = {z:+2} gives p = {sigmoid(z):.4f}")

## 3. Fit the line by minimizing log loss

For a Spam email, the probability assigned to its correct class is $p$. For a Legitimate email, it is $1-p$. Take the negative natural log of that probability, then average across training rows.

$$
  \text{mean binary cross-entropy}
  = -\frac{1}{n}\sum_i[y_i\ln p_i + (1-y_i)\ln(1-p_i)].
$$

A low probability for the recorded label produces a large penalty. For example, a Spam email contributes about 0.105 at $p=0.90$ and 2.303 at $p=0.10$. The implementation below computes the same quantity directly from the score so it still works if the calculated probability rounds to 0 or 1.

The fit uses Newton’s method. Each step uses the slope and curvature of the loss to update the intercept and weight. You can skip reading the optimizer line by line on a first pass. Notice the loss before and after training.

In [ ]:
def loss_from_score(z, y):
    return max(z, 0) - y * z + math.log1p(math.exp(-abs(z)))

def mean_loss(rows, intercept, weight):
    return mean(loss_from_score(intercept + weight * x, y) for _, x, y in rows)

def fit_logistic(rows):
    intercept = weight = 0.0
    history = [mean_loss(rows, intercept, weight)]
    for _ in range(40):
        g0 = g1 = h00 = h01 = h11 = 0.0
        for _, x, y in rows:
            probability = sigmoid(intercept + weight * x)
            error = probability - y
            curvature = probability * (1 - probability)
            g0 += error
            g1 += error * x
            h00 += curvature
            h01 += curvature * x
            h11 += curvature * x * x
        determinant = h00 * h11 - h01 * h01
        step_intercept = (h11 * g0 - h01 * g1) / determinant
        step_weight = (h00 * g1 - h01 * g0) / determinant
        intercept -= step_intercept
        weight -= step_weight
        history.append(mean_loss(rows, intercept, weight))
        if max(abs(step_intercept), abs(step_weight)) < 1e-10:
            break
    return intercept, weight, history

train = splits["train"]
intercept, weight, loss_history = fit_logistic(train)
print(f"Fitted score z = {intercept:.6f} + {weight:.6f} × x")
print(f"Training log loss: {loss_history[0]:.6f} → {loss_history[-1]:.6f}")
print(f"Newton updates: {len(loss_history) - 1}")
print(f"Spam odds multiplier for a one-point increase: {math.exp(weight):.2f}×")

## 4. Work through source emails

Use the fitted score to calculate the model’s Spam probability for E17, which belongs to validation. Its feature is 0.428. The model gives roughly a 48.66% Spam probability. At a cutoff of 0.50, its label is predicted as Legitimate. The recorded label is Spam, so this is a false negative.

Then trace the three training rows from the lesson. The loss uses the probability of the recorded label. In the Legitimate row, use 1 − $p$.

In [ ]:
def score(x):
    return intercept + weight * x

def spam_probability(x):
    return sigmoid(score(x))

validation_by_id = {row[0]: row for row in splits["validation"]}
_, focal_x, focal_y = validation_by_id["E17"]
z = score(focal_x)
p = sigmoid(z)
print(f"E17: z = {intercept:.6f} + {weight:.6f} × {focal_x:g} ≈ {z:.6f}")
print(f"p = 1 / (1 + exp({-z:.6f})) ≈ {p:.6f}")
print("Recorded label:", "Spam" if focal_y == 1 else "Legitimate")

train_by_id = {row[0]: row for row in train}
worked_losses = []
print(f"\n{'Email':<7} {'Feature (%)':>11} {'Label':>10} {'P(Spam)':>10} {'P(recorded)':>12} {'Loss':>8}")
for email in ("E574", "E1748", "E4447"):
    _, x, y = train_by_id[email]
    p = spam_probability(x)
    correct_probability = p if y == 1 else 1 - p
    loss = loss_from_score(score(x), y)
    worked_losses.append(loss)
    label = "Spam" if y == 1 else "Legitimate"
    print(f"{email:<7} {x:>11.3f} {label:>10} {p:>10.4f} {correct_probability:>12.4f} {loss:>8.4f}")
print(f"Mean loss for these three rows: {mean(worked_losses):.4f}")

## 5. Change the decision threshold

The curve maps a measurement to a probability. The threshold maps a probability to a label. Our rule predicts Spam when $p \ge t$, including equality. The model gives the same probability at every threshold.

On the chart, the vertical guide shows where a 0.50 threshold crosses the fitted curve. The chart zooms in on exclamation mark shares between 0% and 2%. Training uses the full input range.

In [ ]:
import matplotlib.pyplot as plt

features = [i * 2 / 200 for i in range(201)]
probabilities = [spam_probability(x) for x in features]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(features, probabilities, color="#176953", linewidth=2, label="Fitted Spam probability")
ax.axhline(0.50, color="#9c542f", linestyle="--", label="0.50 cutoff")
ax.axvline(-intercept / weight, color="#777777", linestyle=":", label="Feature at 50% probability")
ax.scatter([focal_x], [spam_probability(focal_x)], color="#693990", s=65, zorder=5, label="E17")
ax.set(xlim=(0, 2), ylim=(0, 1), xlabel="Exclamation marks (% of email characters)", ylabel="Spam probability")
ax.grid(alpha=0.22)
ax.legend(loc="lower right")
plt.show()

### Compare thresholds using validation emails

A false positive flags a Legitimate message. A false negative lets Spam through. At a lower threshold, the same model can catch more Spam and flag more Legitimate messages. Precision asks what fraction of flags are Spam. Recall asks what fraction of Spam emails are flagged.

In a real system, decide how costly each error is before choosing among validation thresholds. Use 0.50 as the worked operating point for this lesson, or choose a different cutoff using the validation table before you run the final cell.

In [ ]:
def confusion(rows, threshold):
    tp = fp = fn = tn = 0
    for _, x, y in rows:
        flagged = spam_probability(x) >= threshold
        tp += y == 1 and flagged
        fp += y == 0 and flagged
        fn += y == 1 and not flagged
        tn += y == 0 and not flagged
    return tp, fp, fn, tn

def percent(num, denom):
    return f"{num / denom:.1%}" if denom else "undefined"

def report(rows, threshold):
    tp, fp, fn, tn = confusion(rows, threshold)
    print(f"{threshold:>6.2f} {tp:>5} {fp:>5} {fn:>5} {tn:>5}"
          f" {percent(tp, tp + fp):>10} {percent(tp, tp + fn):>9}")

headers = "Cutoff    TP    FP    FN    TN  Precision    Recall"
print("Validation results")
print(headers)
for threshold in (0.35, 0.50, 0.65):
    report(splits["validation"], threshold)

locked_threshold = 0.50
print(f"\nLocked threshold: {locked_threshold:.2f}")

## 6. Report one locked result on the holdout

Run the next cell after making the validation choice. On this benchmark, one punctuation feature can give useful evidence, and some messages still fall on the wrong side. For example, a Legitimate message that scores above the cutoff is a false positive. A Spam message below it is a false negative. Use the Classification Metrics lesson to compare operating points and a model with more features. [Open Classification Metrics](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#classification)

In [ ]:
test = splits["test"]
print(f"Final held-out results on {len(test):,} emails")
print(headers)
report(test, locked_threshold)

test_by_id = {row[0]: row for row in test}
for email in ("E4415", "E19"):
    _, x, y = test_by_id[email]
    p = spam_probability(x)
    actual = "Spam" if y == 1 else "Legitimate"
    predicted = "Spam" if p >= locked_threshold else "Legitimate"
    print(f"{email}: x={x:g}%, p={p:.4f}, predicted {predicted}, recorded {actual}")

## Try it yourself

An email has a score of z = ln(3). Calculate its Spam probability and decide its class at thresholds of 0.50 and 0.80.

<details>
<summary>Check the calculation</summary>

Its Spam odds are $e^{\ln(3)} = 3$. The probability is $3/(1 + 3) = 0.75$. It meets 0.50, so that rule predicts Spam. It falls below 0.80, so that rule predicts Legitimate. The probability stays at 0.75 for both decisions.

</details>